# Continuum channel finding (find_continuum)

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/distributed_applications_tutorials/imaging/find_continuum.ipynb)

Find the continuum channels of a dirty cube as the ALMA pipeline's `hif_findcont` does: a joint mask from the moment 0 and moment 8 images, the mean spectrum in the mask, a channel selection, and the moment-difference iterations that amend the mask (Hunter et al. 2023, *The ALMA Interferometric Pipeline Heuristics*, §6).

`find_continuum` runs the pipeline's `findContinuum.py` (vendored) on an AstroVIPER image store; every reduction of the cube it asks for is a map-reduce over frequency chunks. `find_continuum_fused` gives the same channels in fewer passes over the cube, and `image_cube_single_field(..., find_continuum_first_round=True)` makes the first round while it images the cube, which saves one more pass.

---
## Pseudo Code

```
distributed_applications/imaging/find_continuum(input_image_store, atmospheric_transmission,
                                                atmospheric_sky_temperature, work_directory, ...)
    register the dirty cube, primary beam and point spread function of the store
    mask the cubes where the primary beam is below primary_beam_limit (tclean's pblimit)
    run findContinuum; each immoments / imstat / getprofile of a cube
        -> one map-reduce over frequency chunks (node task find_continuum_cube_reductions)
    return channel selection, warnings, joint mask, summary plot, passes over the cube

find_continuum_fused(...)          # the same results
    a pass also computes the reductions findContinuum asks for next

image_cube_single_field(..., find_continuum_first_round=True)
    every chunk: moment 0 and 8, cube and primary beam statistics, before the write
```

## Notes

- The atmospheric transmission and sky temperature per channel are inputs; the pipeline computes them with CASA's atmosphere model.
- The pipeline gives findContinuum tclean's `.residual`, which has no restoring beams (only the `.psf` has them), so CASA's `immoments` uses its planes as they are, as `find_continuum` does.
- findContinuum changes the working directory and draws with pyplot: a process runs one call at a time.

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    os.system("pip install --upgrade astroviper[all]")

    import astroviper  # noqa: F401 -- availability probe for the pip-install fallback

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

## API

In [ ]:
from astroviper.distributed_applications.imaging import (
    find_continuum,
    find_continuum_fused,
    image_cube_single_field,
)

find_continuum?

## Example 1: a simulated ALMA spectral window

A 480-channel window at 230 GHz observed by 25 ALMA antennas, with a 10 mJy continuum source, a bright compact line at channel 120, a faint extended line at channel 300 and a narrow line at channel 410, and thermal noise.

In [ ]:
import numpy as np

from astroviper.distributed_applications.simulation import simulate_processing_set
from astroviper.utils.beam_models import airy_disk_model
from astroviper.utils.telescope_layout import read_telescope_layout

ARCSEC = np.pi / (180.0 * 3600.0)
PHASE_CENTER = np.array([1.2, -0.4])
N_CHANNELS, N_ANTENNA = 480, 25


def unpolarized(spectrum):
    """XX, XY, YX, YY fluxes of a Stokes I spectrum."""
    zeros = np.zeros_like(spectrum)
    return np.stack([spectrum, zeros, zeros, spectrum], axis=-1)


def line(peak_jy, channel, fwhm_channels):
    sigma = fwhm_channels / np.sqrt(8.0 * np.log(2.0))
    return peak_jy * np.exp(-0.5 * ((np.arange(N_CHANNELS) - channel) / sigma) ** 2)


sky_components = [
    {
        "kind": "point",
        "flux": unpolarized(np.full(N_CHANNELS, 0.01)),
        "ra_dec": PHASE_CENTER,
    },
    {
        "kind": "gaussian",
        "flux": unpolarized(line(0.08, 120, 14)),
        "ra_dec": PHASE_CENTER,
        "major": 1.0 * ARCSEC,
        "minor": 0.8 * ARCSEC,
        "pa": 0.3,
    },
    {
        "kind": "gaussian",
        "flux": unpolarized(line(0.15, 300, 40)),
        "ra_dec": PHASE_CENTER + np.array([4.0, -3.0]) * ARCSEC,
        "major": 6.0 * ARCSEC,
        "minor": 5.0 * ARCSEC,
        "pa": 1.0,
    },
    {
        "kind": "gaussian",
        "flux": unpolarized(line(0.04, 410, 4)),
        "ra_dec": PHASE_CENTER + np.array([-2.0, 2.5]) * ARCSEC,
        "major": 0.6 * ARCSEC,
        "minor": 0.6 * ARCSEC,
        "pa": 0.0,
    },
]
ps_store = simulate_processing_set(
    ps_store="find_continuum_demo.ps.zarr",
    antenna_xds=read_telescope_layout(
        "alma.cycle8.1", antenna_selection=list(range(N_ANTENNA))
    ),
    time_params={
        "time_start": "2021-09-15T04:00:00.000",
        "time_delta": 120.0,
        "n_samples": 20,
    },
    frequency_params={
        "freq_start": 230.0e9,
        "freq_delta": 0.9765625e6,
        "n_channels": N_CHANNELS,
        "channel_width": 0.9765625e6,
        "spectral_window_name": "Band6",
    },
    polarization=["XX", "YY"],
    sky_components=sky_components,
    phase_center_ra_dec=PHASE_CENTER[None, :],
    beam_models=[airy_disk_model("alma")],
    beam_model_map=np.zeros(N_ANTENNA, int),
    noise_params={"random_seed": 7},
    n_time_chunks=1,
    n_frequency_chunks=4,
    overwrite=True,
)["ps_store"]

### The dirty cube, with the first round of find_continuum

`max_iter=0` makes the dirty cube. With `find_continuum_first_round=True` the imager also returns the first-round reductions of `find_continuum_fused` and keeps the variables find_continuum reads.

In [ ]:
from xradio.measurement_set import open_processing_set

image_store = "find_continuum_demo.img.zarr"
imaging = image_cube_single_field(
    ps_store=ps_store,
    image_store=image_store,
    image_params={
        "image_size": [160, 160],
        "cell_size": np.array([-0.3, 0.3]) * ARCSEC,
        "phase_direction": PHASE_CENTER,
        "frequency_coords": open_processing_set(ps_store).xr_ps.get_freq_axis().values,
        "polarization_coords": ["I", "Q"],
        "time_coords": [0],
        "fft_padding": 1.2,
        "cpp_gridder": True,
    },
    imaging_weights_params={
        "weighting": "briggs",
        "robust": 0.5,
        "casa_weighting_implementation": True,
    },
    cycle_control_params={
        "max_iter": 0,
        "max_cycles": 0,
        "threshold": 0.0,
        "gain": 0.1,
        "psf_sidelobe_factor": 1.5,
        "max_iter_per_cycle": -1,
        "min_psf_fraction": 0.05,
        "max_psf_fraction": 0.8,
    },
    image_data_variables_keep=["sky_residual"],
    processing_set_data_group_name="base",
    n_mapping_parallelism={"frequency": 8},
    overwrite=True,
    find_continuum_first_round=True,
)

In [ ]:
# %matplotlib widget  # uncomment for interactive plots (not for headless runs)
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr

cube = xr.open_zarr(image_store, chunks=None)["SKY_RESIDUAL"].sel(polarization="I")[0]
# The brightest pixel of the extended line, around channel 300.
extended = cube.isel(frequency=slice(280, 320)).mean("frequency").values
l_index, m_index = np.unravel_index(np.argmax(extended), extended.shape)
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(cube.isel(l=80, m=80).values * 1e3, label="phase centre")
ax.plot(cube.isel(l=l_index, m=m_index).values * 1e3, label="extended line")
ax.set_xlabel("channel")
ax.set_ylabel("mJy/beam")
ax.legend()
plt.show()

### Continuum channels

The pipeline takes the atmospheric transmission and sky temperature of every channel from CASA's atmosphere model; this window has no atmospheric line, so a flat atmosphere serves here.

In [ ]:
import shutil

from IPython.display import Image, display

atmosphere = {
    "atmospheric_transmission": np.full(N_CHANNELS, 0.92),
    "atmospheric_sky_temperature": np.full(N_CHANNELS, 27.0),
}
shutil.rmtree("find_continuum_demo_work", ignore_errors=True)
separate = find_continuum(
    input_image_store=image_store,
    work_directory="find_continuum_demo_work/separate",
    n_mapping_parallelism={"frequency": 4},
    **atmosphere,
)
print("channel selection:", separate["channel_selection"])
print(
    "all continuum:",
    separate["all_continuum"],
    "; moment difference SNR:",
    round(separate["moment_difference_snr"], 3),
)
display(Image(separate["plot"], width=800))

### Fewer passes over the cube

`find_continuum_fused` finds the same channels in fewer passes; given the imager's first round it skips one more.

In [ ]:
import pandas as pd

fused = find_continuum_fused(
    input_image_store=image_store,
    work_directory="find_continuum_demo_work/fused",
    n_mapping_parallelism={"frequency": 4},
    **atmosphere,
)
with_first_round = find_continuum_fused(
    input_image_store=image_store,
    work_directory="find_continuum_demo_work/imager",
    first_round_reductions=imaging["find_continuum_first_round"],
    n_mapping_parallelism={"frequency": 4},
    **atmosphere,
)
pd.DataFrame(
    {
        name: {
            "channel selection": result["channel_selection"],
            "passes over the store": result["n_cube_reductions"],
            "reads of the dirty cube": result["cube_reads"]["SKY_RESIDUAL"],
            "reads of the primary beam": result["cube_reads"]["PRIMARY_BEAM"],
        }
        for name, result in (
            ("find_continuum", separate),
            ("find_continuum_fused", fused),
            ("fused, first round from the imager", with_first_round),
        )
    }
)